# Aula 3 · Lab 3 — Consumidores, grupos e offsets

**Objetivos:**
- entender que o **grupo de consumidores** divide as partições entre seus membros;
- ver o **rebalanceamento** quando alguém entra ou sai;
- controlar **offsets**: reprocessar do início, a partir de um horário, e medir *lag*.

In [ ]:
import time
from confluent_kafka import Consumer, TopicPartition, OFFSET_BEGINNING
from comum.config import BOOTSTRAP

TOPICO = "aula3-pedidos"

## 1. Um consumidor simples
`poll()` devolve uma mensagem por vez. O consumidor guarda (*commit*) até onde leu no tópico
interno `__consumer_offsets`, por grupo e por partição.

In [ ]:
c = Consumer({"bootstrap.servers": BOOTSTRAP, "group.id": "lab3-notebook",
              "auto.offset.reset": "earliest"})
c.subscribe([TOPICO])

lidas = 0
fim = time.time() + 10
while time.time() < fim and lidas < 15:
    msg = c.poll(1.0)
    if msg is None or msg.error():
        continue
    lidas += 1
    print(f"p{msg.partition()} off={msg.offset()} chave={msg.key()} valor={msg.value()[:50]}")
c.close()

Rode a célula de novo: ela **não** recomeça do zero, porque o grupo `lab3-notebook` já fez commit.

## 2. Grupos e rebalanceamento (no terminal)

1. Abra **3 terminais** no Jupyter. Em cada um:
   ```bash
   cd ~/notebooks && python aula3_kafka/consumidor.py --grupo analise --inicio latest
   ```
   Observe as mensagens `+ recebi partições [...]`: as 6 partições foram divididas entre os 3.
2. Num 4º terminal, gere tráfego:
   ```bash
   cd ~/notebooks && python -c "
   from confluent_kafka import Producer; import os, time
   p = Producer({'bootstrap.servers': os.environ['KAFKA_BOOTSTRAP']})
   for i in range(300):
       p.produce('aula3-pedidos', key=f'cliente{i%12}', value=f'pedido {i}'); p.poll(0); time.sleep(0.1)
   p.flush()"
   ```
3. Mate um consumidor com **Ctrl+C**. O que acontece com as partições dele?
4. Abra um **7º consumidor** no mesmo grupo (com 6 partições). O que ele recebe?
5. Abra um consumidor com `--grupo auditoria`. Ele recebe as mesmas mensagens que o grupo `analise`?

> **Ideia central:** dentro de um grupo, cada partição tem um único dono (escala o processamento).
> Grupos diferentes leem o tópico de forma independente (vários sistemas consomem o mesmo stream).

## 3. Lag: quanto o grupo está atrasado?
Rode no terminal:
```bash
kafka-consumer-groups.sh --bootstrap-server kafka1:9092 --describe --group analise
```
`LAG = LOG-END-OFFSET − CURRENT-OFFSET`. Lag crescendo sem parar quer dizer que o consumidor não acompanha o produtor.
Experimente: `python aula3_kafka/consumidor.py --grupo lento --lento 0.5` enquanto o gerador roda.

In [ ]:
# O mesmo cálculo de lag em Python
def lag_do_grupo(grupo, topico=TOPICO):
    c = Consumer({"bootstrap.servers": BOOTSTRAP, "group.id": grupo, "enable.auto.commit": False})
    parts = [TopicPartition(topico, p) for p in c.list_topics(topico, timeout=10).topics[topico].partitions]
    commitados = c.committed(parts, timeout=10)
    total = 0
    for tp in commitados:
        _, alto = c.get_watermark_offsets(tp, timeout=10)
        atual = tp.offset if tp.offset >= 0 else 0
        print(f"  partição {tp.partition}: commit={atual:<6} fim={alto:<6} lag={alto - atual}")
        total += alto - atual
    c.close()
    print("lag total:", total)

lag_do_grupo("lab3-notebook")

## 4. Reprocessando (replay)
O log do Kafka é **retido**: dá para voltar no tempo. Isso é a base da arquitetura Kappa.

### 4a. Do início, sem mexer no grupo: `assign()` manual

In [ ]:
c = Consumer({"bootstrap.servers": BOOTSTRAP, "group.id": "lab3-replay", "enable.auto.commit": False})
c.assign([TopicPartition(TOPICO, 0, OFFSET_BEGINNING)])
for _ in range(5):
    m = c.poll(5.0)
    if m and not m.error():
        print(m.partition(), m.offset(), m.value()[:40])
c.close()

### 4b. A partir de um horário: `offsets_for_times()`
Cada mensagem tem um timestamp. Vamos começar a ler a partir de 5 minutos atrás.

In [ ]:
c = Consumer({"bootstrap.servers": BOOTSTRAP, "group.id": "lab3-replay", "enable.auto.commit": False})
cinco_min_atras_ms = int((time.time() - 300) * 1000)
parts = [TopicPartition(TOPICO, p, cinco_min_atras_ms)
         for p in c.list_topics(TOPICO, timeout=10).topics[TOPICO].partitions]
offsets = c.offsets_for_times(parts, timeout=10)
for tp in offsets:
    print(f"partição {tp.partition}: começar no offset {tp.offset}")
c.assign(offsets)
m = c.poll(5.0)
print("primeira mensagem:", m.value()[:60] if m and not m.error() else None)
c.close()

### 4c. Resetando o grupo inteiro (com o grupo parado)
```bash
kafka-consumer-groups.sh --bootstrap-server kafka1:9092 --group analise --topic aula3-pedidos \
    --reset-offsets --to-earliest --dry-run      # mostra o que faria
kafka-consumer-groups.sh --bootstrap-server kafka1:9092 --group analise --topic aula3-pedidos \
    --reset-offsets --to-datetime 2026-01-01T00:00:00.000 --execute
```

## Exercícios
1. Desligue o auto-commit (`enable.auto.commit=False`) e faça commit manual (`c.commit(msg)`) a cada 10 mensagens.
   Mate o processo no meio: quantas mensagens serão relidas? Que semântica é essa (at-least-once ou at-most-once)?
2. E se o commit fosse feito **antes** de processar a mensagem?
3. Com o gerador da loja rodando (`python -m comum.gerador --taxa 20`), escreva um consumidor do tópico `pedidos`
   que imprima a receita acumulada por cidade a cada 10 segundos.
   Quando você reiniciar o consumidor, o total recomeça do zero. Por quê? Guarde esse problema: o Spark resolve isso com **estado + checkpoint**.